# PAAC-VN · Chặng 02 · E5 trên Kaggle

Thêm **Private Dataset** từ gói paac_vn_stage02_kaggle.zip, bật **GPU** và **Internet**, rồi chạy lần lượt.
Gói đã có hai dataset và sáu embeddings lịch sử. Notebook gọi code đã commit, giữ nguyên sáu cells và FP32/batch8.
**Run All mặc định chỉ chạy preflight/smoke; full bị bỏ qua đến khi có phê duyệt của tác giả.**


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, zipfile, shutil

# Để trống để tự tìm gói duy nhất trong Add Input (ZIP hoặc thư mục Kaggle đã giải nén).
INPUT_PATH = ""
INPUT_BASE = Path("/kaggle/input")
WORK_BASE = Path("/kaggle/working")
SESSION_TAG = datetime.now(timezone.utc).strftime("kg_%Y%m%d_%H%M%S")
REPO = WORK_BASE / "PAAC-VN"
PREFLIGHT_ID, SMOKE_ID, FULL_ID = [SESSION_TAG + s for s in ("_preflight", "_smoke", "_full")]
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["HF_HOME"] = str(WORK_BASE / "paac_hf_cache")
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"
print("Session:", SESSION_TAG)


## 1. Khôi phục code đúng commit và đầu vào

Không cần tài khoản GitHub. Git bundle giữ nguyên commit thật; payload được kiểm tra SHA-256 trước khi dùng.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

if INPUT_PATH:
    input_path = Path(INPUT_PATH)
else:
    candidates = list(INPUT_BASE.rglob("bundle_manifest.json"))
    if len(candidates) == 1:
        input_path = candidates[0].parent
    else:
        candidates = list(INPUT_BASE.rglob("paac_vn_stage02_kaggle.zip"))
        if len(candidates) != 1:
            raise RuntimeError("Hãy Add Input chứa một gói Stage02 hoặc điền INPUT_PATH cụ thể.")
        input_path = candidates[0]

if input_path.is_file():
    unpacked = WORK_BASE / "stage02_input"
    if not unpacked.exists():
        with zipfile.ZipFile(input_path) as archive:
            for item in archive.infolist():
                target = (unpacked / item.filename).resolve()
                if not target.is_relative_to(unpacked.resolve()):
                    raise RuntimeError("Đường dẫn ZIP không hợp lệ.")
            archive.extractall(unpacked)
    bundle_root = unpacked
else:
    bundle_root = input_path

manifest = json.loads((bundle_root / "bundle_manifest.json").read_text())
assert sha256(bundle_root / "repo.bundle") == manifest["git_bundle_sha256"], "Sai Git bundle hash"
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout", str(bundle_root / "repo.bundle"), str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", manifest["commit"]], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == manifest["commit"], "Repo hiện tại thuộc commit khác; dùng working directory mới."
for name, digest in manifest["source_sha256"].items():
    assert sha256(REPO / name) == digest, f"Code/config khác gói: {name}"
for item in manifest["payload"]:
    source = bundle_root / "payload" / item["path"]
    assert sha256(source) == item["sha256"], f"Sai payload hash: {item['path']}"
    target = REPO / item["path"]
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        try:
            target.symlink_to(source.resolve())
        except OSError:
            shutil.copyfile(source, target)
    assert sha256(target) == item["sha256"], f"Đầu vào hiện tại khác gói: {item['path']}"
print("Đã khôi phục", actual_commit, "| payload:", len(manifest["payload"]), "files")


## 2. Môi trường GPU

Giữ PyTorch CUDA có sẵn của Kaggle; dependency bổ sung nằm trong venv riêng. Không cài requirements-lock lịch sử lên image GPU.
Không cài lại hoặc đổi GPU/package giữa smoke và full.


In [ ]:
base_runtime = json.loads(subprocess.check_output([
    sys.executable, "-c",
    "import torch,json; print(json.dumps({'torch':torch.__version__, 'cuda':torch.cuda.is_available(), 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"
], text=True))
if not base_runtime["cuda"]:
    raise RuntimeError("Bật Settings → Accelerator → GPU, khởi động session GPU rồi chạy lại từ đầu.")
print(base_runtime)
VENV = WORK_BASE / "paac-e5-venv"
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV)], check=True)
ENV_DIR = REPO / "results/e5_stage02" / (SESSION_TAG + "_launcher")
ENV_DIR.mkdir(parents=True, exist_ok=True)
requirements = REPO / "kaggle/requirements.txt"
marker = VENV / "stage02_setup.json"
setup_key = {"requirements_sha256": sha256(requirements), "base_torch": base_runtime["torch"]}
if marker.exists():
    assert json.loads(marker.read_text()) == setup_key, "Environment cũ khác requirements/torch; cần working session mới."
else:
    constraints = VENV / "torch_constraint.txt"
    constraints.write_text("torch==" + base_runtime["torch"] + "\n")
    subprocess.run([str(PYTHON), "-m", "pip", "--disable-pip-version-check", "install",
                    "-c", str(constraints), "-r", str(requirements),
                    "--log", str(ENV_DIR / "dependency_install.log")], check=True)
    marker.write_text(json.dumps(setup_key))
check = json.loads(subprocess.check_output([
    str(PYTHON), "-c",
    "import torch,json; print(json.dumps({'torch':torch.__version__, 'cuda':torch.cuda.is_available()}))"
], text=True))
assert check["cuda"] and check["torch"] == base_runtime["torch"], "PyTorch GPU đã thay đổi; dừng."
freeze = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze", "--all"], text=True)
(ENV_DIR / "environment_freeze.txt").write_text(freeze)
(ENV_DIR / "base_runtime.json").write_text(json.dumps(base_runtime, indent=2))
print("GPU environment sẵn sàng:", check)


## 3. Preflight và smoke

Smoke encode hai lần trên 500 apps/cell, dùng hai lần load model riêng. Token equality audit vẫn chạy toàn bộ paired cohort.
Lần đầu cần Internet để tải checkpoint pinned. Log và failure receipts cũng được lưu nếu smoke lỗi.


In [ ]:
from IPython.display import display, FileLink
RESULTS = REPO / "results/e5_stage02"

def export_results(kind):
    archive_path = WORK_BASE / ("stage02_" + kind + "_results.zip")
    if archive_path.exists():
        archive_path = WORK_BASE / ("stage02_" + kind + "_results_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + ".zip")
    folders = [RESULTS / PREFLIGHT_ID, RESULTS / SMOKE_ID, ENV_DIR]
    if kind == "full":
        folders.append(RESULTS / FULL_ID)
    with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED, compresslevel=1) as zipped:
        for folder in folders:
            if folder.exists():
                for path in sorted(folder.rglob("*")):
                    if path.is_file():
                        zipped.write(path, path.relative_to(REPO).as_posix())
    print("Output:", archive_path, "| MB:", round(archive_path.stat().st_size / 1e6, 1))
    display(FileLink(str(archive_path)))
    return archive_path

def run_stage(mode, run_id, extra=()):
    log_path = ENV_DIR / (run_id + ".log")
    command = [str(PYTHON), "-B", "-m", "pipeline.e5_stage02", mode, run_id, *map(str, extra)]
    print("Chạy:", " ".join(command), flush=True)
    with log_path.open("x") as log:
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1,
                                   env=os.environ.copy())
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f"{mode} dừng (exit={returncode}); xem {log_path.name} và execution_receipt.json")
    receipt = json.loads((RESULTS / run_id / "execution_receipt.json").read_text())
    return receipt

preflight = run_stage("preflight", PREFLIGHT_ID)
assert preflight["status"] == "READY_FOR_SMOKE"
print("READY_FOR_SMOKE")


In [ ]:
SMOKE_RECEIPT = RESULTS / SMOKE_ID / "execution_receipt.json"
try:
    smoke = run_stage("smoke", SMOKE_ID, ["--allow-download"])
    assert smoke["status"] == "SMOKE_PASS"
    print("\nCell | max abs difference | minimum pair cosine")
    for cell in smoke["cells"]:
        print("/".join(cell["cell"]), "|", cell["comparison"]["max_abs_difference"],
              "|", cell["comparison"]["min_pair_cosine"])
    print("SMOKE_PASS — chưa phê duyệt full.")
finally:
    export_results("smoke")


## 4. Phê duyệt full — mặc định bỏ qua

Dừng ở đây để tác giả xem smoke. Chỉ sau phê duyệt thật mới nhập tên/tham chiếu và đặt True.
Không sửa scientific contract; authorization riêng sẽ được gắn SHA-256 của smoke receipt và đúng full run ID.
Sau khi điền, chạy lại cell dưới và cell full ngay sau đó trong **cùng session/GPU/environment**.


In [ ]:
AUTHOR_APPROVED_FULL = False
APPROVED_BY = ""
APPROVAL_REFERENCE = ""  # Ngày + nội dung xác nhận của tác giả, sau khi xem smoke.

FULL_AUTHORIZATION = ENV_DIR / "full_authorization.json"
if AUTHOR_APPROVED_FULL:
    assert APPROVED_BY.strip() and APPROVAL_REFERENCE.strip(), "Cần tên người duyệt và bằng chứng phê duyệt."
    smoke = json.loads(SMOKE_RECEIPT.read_text())
    assert smoke["status"] == "SMOKE_PASS"
    authorization = {
        "full_run_authorized": True, "approved_by": APPROVED_BY.strip(),
        "approval_reference": APPROVAL_REFERENCE.strip(), "full_run_id": FULL_ID,
        "source_binding": smoke["source_binding"],
        "smoke_receipt_sha256": sha256(SMOKE_RECEIPT),
    }
    if FULL_AUTHORIZATION.exists():
        assert json.loads(FULL_AUTHORIZATION.read_text()) == authorization, "Không ghi đè approval khác."
    else:
        with FULL_AUTHORIZATION.open("x") as handle:
            json.dump(authorization, handle, indent=2)
    print("Đã ghi xác nhận tác giả cho", FULL_ID)
else:
    print("Chưa được phê duyệt: KHÔNG tạo authorization và KHÔNG chạy full.")


In [ ]:
if AUTHOR_APPROVED_FULL:
    # Kiểm tra đầu vào lịch sử trước khi dùng GPU cho full.
    historical = [item for item in manifest["payload"] if item["path"].endswith(".npy")]
    assert len(historical) == 6
    for item in historical:
        assert sha256(REPO / item["path"]) == item["sha256"], item["path"]
    try:
        full = run_stage("full", FULL_ID, ["--smoke-receipt", SMOKE_RECEIPT,
                                          "--full-authorization", FULL_AUTHORIZATION])
        assert full["status"] == "FULL_COMPLETE"
        print("FULL_COMPLETE:", len(full["cells"]), "cells, đã lưu validation và historical comparisons.")
    finally:
        export_results("full")
else:
    print("Bỏ qua full. Tải stage02_smoke_results.zip và xin phê duyệt trước.")


## Giữ kết quả

Tải ZIP từ link ở trên hoặc mục **Output → /kaggle/working**. Không cần tải venv hay checkpoint cache.
Nếu smoke/full lỗi, ZIP vẫn có log và receipt của phần đã chạy; không tự giảm batch/đổi precision/tắt determinism.
Notebook chưa được chạy trên GPU trong phiên local; kết quả Kaggle là bằng chứng thực thi cần thu về cho Chặng 02.
